In [0]:
%run "../utils/03_write_to_delta_utils"

from pyspark.sql.functions import col, when

#-------------------------------------------
#    dim_products dataframe
#-------------------------------------------
df_products_silver = spark.read.table("novamart.silver.sql_products")

df_dim_products = (df_products_silver
                   .select(
                       col("product_id"),
                       col("product_name"),
                       col("category"),
                       col("subcategory"),
                       col("brand"),
                       col("cost_price"),
                       col("unit_price"),
                       col("supplier_id"),
                       col("reorder_threshold"),
                       col("is_margin_positive"),
                       when((col("is_margin_positive") == True) & (col("unit_price") > 0),
                            ((col("unit_price") - col("cost_price")) / col("unit_price")) * 100)
                       .otherwise(0.00).cast("decimal(5,2)").alias("gross_margin_percentage"),
                       col("created_date").alias("product_created_at")
                   ))

#-------------------------------------------
#   write to delta table dim_products
#-------------------------------------------

write_delta_table(
    df = df_dim_products,
    table_name = "novamart.gold.dim_products",
    write_mode = "merge",
    merge_key = "product_id",
    cluster_keys= ["product_id"]
)
#-------------------------------------------

#-------------------------------------------
#    fact_inventory dataframe
#-------------------------------------------
df_fact_inventory = (df_products_silver
                     .select(
                         col("product_id"),
                         col("stock_quantity"),
                         col("is_reorder_needed")
                     ))
